- 평가의 목적은 위험을 줄이고 기회를 발견
- 시스템 컨텍스트 고려가 필요
  - 시스템이 어떤 부분에서 취약한지 파악해 이에 맞게 평가 설계

## 3.1. 파운데이션 모델 평가의 어려움
- AI 모델이 똑똑해질수록 평가가 더 어려움
  - 정교한 작업일수록 평가에 훨씬 더 많은 시간 소요됨
- 개방형 특성으로 인해 정답을 기준으로 성능을 평가하는 기존 방식은 더 이상 유효하지 않음
- 모델 제공업체가 모델 세부 사항을 공개하지 않거나, 애플리케이션 개발자가 이를 이해할 지식이 부족
  - 출력 결과만 보고 평가할 수 밖에 없음
- 파운데이션 모델의 성능이 너무 빠르게 향상되어, 일반 언어 평가 (GLUE), NaturalInstructions, MMLU 등 공개적으로 이용 가능한 벤치마크들은 파운데이션 모델을 평가하는 데 부적절한 것으로 판명
- 범용 모델의 평가 범위가 확장
  - 모델이 수행할 수 있는 새로운 작업 발견하는 것도 포함
- 투자가 불충분해 불충분한 인프라로 체계적인 평가 수행을 어렵게 함

## 3.2. 언어 모델링 지표 이해하기
- 언어 모델은 파운데이션 모델의 핵심 구성 요소로, 언어 모델의 성능이 좋을수록, 파운데이션 모델 성능이 좋은 경향이 있음
- 자기회귀 언어 모델 포함 토큰들로 이루어진 시퀀스를 생성하는 모델은 교차 엔트로피나 관련 지표인 퍼플렉시티 사용해 학습
  - 교차 엔트로피의 변형된 형태로 문자당 비트 (BPC), 바이트당 비트 (BPB)가 있음
- 모델이 학습 데이터의 분포를 잘 학습할수록 학습 데이터에서 다음에 올 것을 더 잘 예측하고 학습 교차 엔트로피 낮아짐
- 일반적으로 운영 환경의 데이터가 학습 데이터와 비슷할수록 모델이 더 좋은 성능을 냄

### 3.2.1. 엔트로피
- 토큰이 평균적으로 얼마나 많은 정보를 담고 있는지 측정
- 엔트로피가 높을수록 각 토큰이 더 많은 정보를 담고 있고, 토큰을 표현하는 데 더 많은 비트 필요
- 언어에서 다음에 올 것을 예측하기가 얼마나 어려운지 보여줌
- $$H(X) = - \sum_{i} p(x_i) \log p(x_i)$$
  - $X$는 확률변수로 언어 모델에서의 다음 토큰 의미
  - $x_i$는 확률변수 $X$가 가질 수 있는 값으로 언어 모델에서의 어휘
  - $\log p(x_i)$는 해당 사건의 정보량
    - 정보량은 **"그 사건을 알게 되었을 때, 내가 새로 얻은 지식의 양"**을 의미
    - 다른 표현으로는, **기존 믿음의 수정량**을 뜻함
    - 이에 따라 확률이 높을록 정보량 작고, 확률이 낮을수록 정보량 큼
    - 항상 음수이기 때문에 $-$가 붙음
  - 전체 수식의은 평균 정보량을 의미

### 3.2.2. 교차 엔트로피
- 언어 모델이 데이터셋의 내용을 얼마나 예측하기 어려워하는지 보여주는 지표
- 아래 두 가지 특성에 따라 달라짐
  - 학습 데이터의 예측 가능성
    - 학습 데이터의 엔트로피
    - $H(P)$
  - 언어 모델이 파악한 분포가 학습 데이터의 실제 분포와 얼마나 다른지
    - 쿨백 - 라이블리 (KL) 발산으로 측정
      -  쿨백 - 라이블리 (KL) 발산은 "두 확률분포가 얼마나 다른가"를 측정
- $$H(p,q) = H(p) + D_{KL}(p \parallel q)$$
  - $H(p,q) = H(p) + \sum_x p(x)\log\frac{p(x)}{q(x)}$
  - $H(p,q) = H(p) + \mathbb{E}_{p}\left[(-\log q(x)) - (-\log p(x))\right]$
    - $-\log q(x)$은 모델의 정보량을 의미
    - $-\log p(x)$은 최적에서의 정보량
    - $\mathbb{E}_{p}\left[(-\log q(x)) - (-\log p(x))\right]$은 실제 분포 기준 모델 정보량의 평균을 의미
- 비대칭적
  - P에 대한 Q의 교차 엔트로피는 Q에 대한 P의 교차 엔트로피와 서로 다른 값을 가짐
- 언어 모델은 학습 데이터에 대한 교차 엔트로피를 최소화하도록 학습

### 3.2.3. 문자당 비트와 바이트당 비트
- 단위로 비트 사용
  - 교차 엔트로피에서 비트는 각 토큰을 표현하는 데 필요한 비트를 의미
  - 모델마다 토큰화 방식이 달라 토큰당 비트 수로 모델 비교가 어려움
- 문자당 비트 (bits-per-character, BPC) 사용
  - 토큰당 비트 수를 토큰을 이루는 문자 수로 나눔
  - 문자 인코딩 방식이 다양하다는 문제점이 있음
- 바이트당 비트 (bits-per-byte, BPB) 사용
  - 원본 학습 데이터 1바이트를 표현하는 데 필요한 비트 수 의미
  - 언어 모델이 텍스트를 얼마나 효율적으로 압축할 수 있는지 알려줌
    - 원본 1바이트 (8비트) 대비 BPB 값으로 알 수 있음

### 3.2.4. 퍼플렉시티
- PPL로 줄여 씀
- 엔트로피와 교차 엔트로피의 지수 함수
- 교차 엔트로피는 모델이 다음 토큰을 예측하기 얼마나 어려운지 측정
- 퍼플렉시티는 다음 토큰을 예측할 때 불확실성 측정
  - 교차 엔트로피와 퍼플렉시티는 동일한 정보를 담고 있음
  - 퍼플렉시티는 로그 스케일에 있던 교차 엔트로피 값을 지수를 취함으로써 다른 해석 취함
  - 교차 엔트로피는 학습 관점의 난이도를 의미
  - 퍼플렉시티는 불확실성 관점에서 선택지의 수를 의미
- 실제 분포 P를 가진 데이터셋의 퍼플렉시티
  - $$\mathrm{PPL}(P) = 2^{H(P)}$$
- 데이터셋에서 언어 모델 (학습된 분포 Q)의 퍼플렉시티
  - $$\mathrm{PPL}(P, Q) = 2^{H(P, Q)}$$
- 교차 엔트로피 단위로 비트를 사용하며, 비트는 2개의 고유한 값을 표현할 수 있어 밑으로 2 사용
- ML 프레임워크에서는 엔트로피와 교차 엔트로피 단위로 nat 사용
  - 자연로그 기반 정보 단위
  - 수학적, 계산 상 편의성 때문에 사용

### 3.2.5. 퍼플렉시티 해석과 활용 사례
- 데이터 자체의 특성과 퍼플렉시티 어떻게 계산하는지에 따라 퍼플렉시티 값 달라짐
  - 구조화된 데이터일수록 퍼플렉시티가 낮음
  - 어휘 크기가 클수록 퍼플렉시티 높음
  - 컨텍스트 길이가 길수록 퍼플렉시티가 낮음
- 퍼플렉시티는 모델의 성능을 간접적으로 보여주는 좋은 지표
  - 모델이 다음 토큰을 잘 예측하지 못하면 다른 작업에서도 성능이 좋지 않을 가능성이 높기 때문
- 퍼플렉시티는 어떤 텍스트가 모델의 학습 데이터에 포함되었는지 탐지하는 데 사용 가능
  - 모델이 학습 중에 본 적 있고 기억하는 텍스트에서는 퍼플렉시티가 가장 낮게 나타나기 때문
- 비정상적인 텍스트를 탐지하는 데 퍼플렉시티가 사용될 수 있음
  - 특이한 생각을 표현하는 텍스트나 의미 없는 텍스트처럼 예측하기 어려운 텍스트에서 퍼플렉시티 가장 높게 나타나기 때문
- 사후 학습된 모델 평가하는 데 퍼플렉시티가 적절하지 않을 수 있음
  - 사후 학습은 특정 작업을 잘 수행하도록 가르치는 과정이기 때문에, 다음 토큰 예측하는 능력은 떨어질 수 있기 때문
- 모델의 수치 정밀도와 메모리 사용량을 줄이는 양자화 기법도 퍼플렉시티를 변화시킬 수 있음

## 3.3. 정확한 평가
- 정확한 평가는 모호함 없이 내리는 판단을 의미
- 기능적 정확성 (functional correctness)과 참조 데이터의 유사도 측정 두 가지 방식 존재

### 3.3.1. 기능적 정확성
- 시스템이 의도한 기능을 제대로 수행하는지 평가하는 것을 의미
- 항상 간단하지 않으며, 측정을 자동화하기도 어려움
- 전체 문제 중 해결한 문제의 비율인 pass@k로 측정 가능
  - 문제마다 k개의 응답을 생성해, k 중 하나라도 해당 문제의 모든 테스트 케이스를 통과하면 그 문제를 해결한 것으로 봄
- 측정 가능한 목표가 있는 작업은 보통 기능적 정확성으로 평가할 수 있음
  - e.g. 게임봇이 획득한 점수로 성능을 알 수 있음

### 3.3.2. 참조 데이터 유사도 측정
- AI 출력을 참조 데이터와 비교해 평가하는 방법
  - 참조가 필요한 평가 지표는 참조 기반 지표 (reference-based metrics), 그렇지 않은 것은 참조 없는 지표 (reference-free metrics)라고 함
- 참조 데이터는 정답이나 표준 응답이라고도 불림
  - 참조 데이터는 (입력, 참조 응답) 형식을 따름
  - 하나의 입력에 여러 개의 참조 응답이 있을 수 있음
- 참조 데이터를 얼마나 빨리 많이 만들 수 있는지가 성능의 척도가 됨
  - 보통 사람이 생성하지만 점점 AI가 생성하는 경우가 늘고 있음
- 생성된 응답이 참조 응답과 더 비슷할수록 더 좋은 것으로 간주
  - 두 개방형 텍스트 간 유사도를 측정하는 방법은 4가지가 있음
    - 비교
      - 평가자에게 두 텍스트가 같은지 판단하도록 요청
    - 정확한 일치
      - 생성된 응답이 참조된 응답 중 하나와 정확하게 일치하는지 여부
      - 점수가 이진 (일치 또는 불일치)
    - 어휘적 유사도
      - 생성된 응답이 참조 응답과 얼마나 비슷해 보이는지
      - 연속적인 척도 (0과 1사이 또는 -1과 1 사이)
    - 의미적 유사도
      - 생성된 응답이 의미에서 참조 응답과 얼마나 가까운지
      - 연속적인 척도 (0과 1사이 또는 -1과 1 사이)

#### 정확한 일치
- 생성된 응답이 참조 응답 중 하나와 정확히 일치하면 정확한 일치로 간주
- 표현 방식이 다를 수 있어 참조 답안이 포함된 모든 응답을 정답으로 인정할 수 있음
  - 잘못된 답을 정답으로 인정할 수 있음
- 간단한 작업을 넘어서면 거의 작동하지 않음
  - 하나의 입력에 대해 가능한 모든 응답을 만드는 것은 불가능해, 복잡한 작업에서는 어휘적 유사도와 의미적 유사도가 더 효과적

#### 어휘적 유사도
- 각 텍스트를 더 작은 토큰으로 나누어 비교해, 두 텍스트가 얼마나 겹치는지 측정
  - 두 텍스트가 비슷하게 생겼는지 측정할 뿐 의미가 같은지 측정하지 않음
- 어휘적 유사도를 측정하는 방법
  - 근사 문자열 매칭 (approximate string matching, 퍼치 매칭 fuzzy matching)
    - 한 텍스트를 다른 텍스트로 바꾸는 데 필요한 편집 횟수 (편집 거리 edit distance) 측정
  - n-gram 유사도
    - 토큰의 연속된 시퀀스인 n-gram의 겹침을 기준으로 측정
    - 참조 응답 n-gram 중 몇 퍼센트가 생성된 응답에도 있는지 측정
- 일반적인 어휘적 유사도 지표 BLEU, ROUGE, METEOR++, TER, CIDEr
  - 이런 지표를 사용하는 벤치마크 WMT, COCO Captions, CEMv2
  - 파운데이션 모델 등장 이후 어휘적 유사도 사용 벤치마크가 줄어듦
- 단점
  - 포괄적인 참조 응답 세트를 만들어야 함
    - 참조 세트에 비슷한 응답이 없으면 좋은 응답도 낮은 유사도 점수를 받을 수 있기 때문
  - 참조 응답의 품질이 낮을 수 있음
  - 어휘적 유사도가 높다고 반드시 더 나은 응답이라고 할 수 없음

#### 의미적 유사도
- 텍스트를 임베딩이라고 부르는 숫자 표현으로 변환해, 의미가 얼마나 비슷한지 계산
- 지표로 BERT가 임베딩을 생성하는 BERTScore, 여러 알고리즘 조합으로 임베딩을 생성하는 MoverScore가 있음
- 단점
  - 기반이 되는 임베딩 알고리즘의 품질에 따라 신뢰성이 결정
  - 기반이 되는 알고리즘을 실행하는 데 상당한 계산 능력과 시간이 필요

### 3.3.3. 임베딩 소개
- 원본 데이터의 의미를 담으려는 숫자 표현 
- 보통 100에서 10,000 크기의 벡터
- 임베딩을 만들기 위해 학습된 모델로 BERT, CLIP (대조적 언어-이미지 사전 학습), Sentence Transformer, API 형태로 제공되는 비공개 임베딩 모델이 있음
- GPT 등 ML 모델은 보통 입력을 벡터로 표현해야 하기 때문에 임베딩을 생성하는 단계를 포함하고, 이런 모델의 중간 층에 접근할 수 있으면 임베딩을 추출할 수 있지만, 임베딩 전용 모델이 생성한 임베딩만큼 좋지 않을 수 있음
- 평가
  - 더 비슷한 텍스트의 임베딩이 코사인 유사도나 관련 지표로 측정했을 때 더 가까우면 좋은 임베딩 알고리즘이라고 봄
  - MTEB (Massive Text Embedding Benchmark) 벤치마크처럼 작업의 유용성을 기준으로도 평가 가능
- 서로 다른 유형의 데이터에 대한 통합 임베딩을 만드는 연구도 존재
  - 여러 유형의 데이터를 표현할 수 있는 통합 임베딩 공간은 멀티모달 임베딩 공간이라고 함

## 3.4. AI 평가자
- AI를 사용해 AI를 평가하는 접근 방식을 AI 평가자 (AI as a judge) 또는 LLM 평가자 (LLM as a judge)라고 함
- 다른 AI 모델을 평가하는 데 사용되는 AI 모델을 AI 평가자라고 함
- AI 모델의 성능이 개선되며 AI 평가자는 AI 모델을 평가하는 보편적인 방법이 됨

### 3.4.1. AI 평가자를 쓰는 이유
- 사람 평가자에 비해 빠르고, 사용하기 쉽고, 비용도 상대적으로 저렴
- 참조 데이터 없이도 작동해 실제 서비스 환경에서도 사용 가능
- 많은 사람의 의견을 바탕으로 만들어져 대중적인 관점에서 판단을 내릴 수 있음
- 사람 평가자들과 높은 상관관계 보이기도 함
- 자신의 결정을 설명할 수 있어 평가 결과를 검토하고 싶을 때 유용

### 3.4.2. AI 평가자 사용법
- AI를 사용해 응답 자체의 품질을 평가하거나, 그 응답을 참조 데이터와 비교하거나, 다른 응답과 비교할 수 있음
- AI 도구가 제공하는 내장 AI 평가로는 Azure AI Foundry, MLflow.metrics, LangChain Criteria Evaluation, Ragas 존재
  - 표준화되어 있지 않음
- 평가자 프롬프트는 아래 사항을 명확하게 설명해야 함 
  - 모델이 수행할 작업
  - 모델을 평가할 때 따라야 할 기준
  - 점수 체계
- 언어 모델은 숫자보다 텍스트를 더 잘 다루어 수치 점수 체계보다 분류에서 더 잘 작동
- 연속 점수보다 이산 점수가 더 잘 작동하며, 이산 점수 범위가 넓을수록 모델 성능 더 나빠짐
- AI 평가자는 모델과 프롬프트 모두 포함하는 시스템으로, 모델, 프롬프트 또는 모델의 샘플링 파라미터를 변경하면 다른 평가자가 됨

### 3.4.3. AI 평가자의 한계
- 동어반복처럼 보이기도 함
- 확률적 특성으로 신뢰도가 떨어진다고 볼 수 있음
- 상당한 비용과 지연을 초래할 위험도 있음

#### 비일관성
- AI 평가자는 확률적
- 샘플링 변수로 일관성을 높일 순 있음
- 높은 일관성이 높은 정확도를 의미하지 않을 순 있음

#### 평가 기준의 모호성
- AI 평가자 지표는 표준화되지 않아 잘못 해석하고 오용하기 쉬움
- 오픈 소스 도구들이 동일한 기준에 대해 서로 다른 지시와 점수 체계를 갖고 있음
  - 동일한 기준에 대해 비교할 수 없음
- 시간이 지남에 따라서도 변할 수 있음
- AI 평가자를 신뢰하기 위해서는 모델과 평가자에 사용된 프롬프트를 확인해야 함

#### 비용과 지연 시간 증가
- 강력한 모델로 응답 평가하는 것은 비용이 많이 들 수 있음
- 평가자로 더 약한 모델을 사용하거나 표본 검사 (응답의 일부만 평가)로 비용을 절감할 수 있음
  - 표본 검사는 일부 실패를 놓칠 수 있음
- 운영 파이프라인에 AI 평가자 구현할 경우 위험은 줄지만 지연 시간이 늘어날 수 있음

#### AI 평가자의 편향
- AI 평가자에도 편향이 있으며, 평가자마다 서로 다른 편향 보임
- 자기 편향
  - 모델이 다른 모델이 생성한 응답보다 자신의 응답을 선호하는 현상
- 첫 위치 편향
  - 여러 선택지 중 첫 번째 응답 선호
  - 순서를 바꿔가며 테스트를 여러 번 반복하거나 세심하게 작성된 프롬프르를 사용해 완화
  - 사람은 마지막에 본 응답을 선호하는 경향 (최근성 편향, recency bias)이 있음
- 이외에도 개인정보 보호와 지적 재산권 제약으로 평가가 어려워지기도 함

### 3.4.4. 평가자로 활용 가능한 모델
- 자기 편향 때문에 편법처럼 보일 수 있지만, 자기 평가나 자기 비평은 기본 검증에 유용
- 평가 받는 모델보다 더 강력한 모델은 더 나은 판단을 할 수 있고, 더 나은 응답을 생성하도록 안내해 성능 향상에 도움을 줄 수 있음
  - 비용과 지연 시간 때문에 응답 생성에 더 강력한 모델을 사용하지 못할 수 있음
  - 문제점
    - 가장 강력한 모델을 평가할만한 평가자를 찾을 수 없음
    - 어떤 모델이 가장 강력한지 판단하기 위한 다른 평가 방법이 필요
  - 범용 평가자에 한해 더 강력한 모델이 사람의 선호도와 상관관계가 있음
- 특정 기준과 특정 점수 체계를 사용해 특정 판단을 하도록 학습된, 작고 특화된 평가자는 범용적인 평가자보다 신뢰할 수 있음
  - 보상 모델
    - (프롬프트, 응답) 쌍을 입력 받고 주어진 프롬프트에 대해 그 응답이 얼마나 좋은지 점수를 매긺
    - e.g. 구글의 캐피 (Cappy)
  - 참조 기반 평가자
    - 하나 이상의 참조 응답을 기준으로 생성된 응답 평가
    - 유사도 점수나 품질 점수 (생성된 응답이 참조 응답과 비교해 얼마나 좋은지)를 출력 가능
    - e.g. BLEURT, Prometheus
  - 선호도 모델
    - (프롬프트, 응답 1, 응답 2)를 입력으로 받아 주어진 프롬프트에 대해 어느 응답이 더 나은지 (사용자가 더 선호하는지) 출력
    - e.g. JudgeLM, PandaLM

## 3.5. 비교 평가를 통해 모델 순위 정하기
- 어떤 모델이 가장 적합한지 알고 싶은 경우 **모델 순위** 사용
- 모델 순위는 개별 평가나 비교 평가를 통해 결정
- 개별 평가는 각 모델을 독립적으로 평가한 다음 점수를 기준으로 순위를 매김
- 비교 평가는 모델들을 서로 비교하고 비교 결과로 순위 계산
  - 모델 간 비교를 경기 (match)라고 부름
  - 비교 평가 결과 주어지면 평가 알고리즘 사용해 모델 순위 계산

### 3.5.1. 비교 평가의 과제들
- 점수 기반 평가에서는 벤치마크와 지표 설계가 가장 어려움
- 비교 평가에서는 비교 결과 수집하는 것과 모델 순위 매기는 것 모두 까다로움
- 아래는 비교 평가의 3가지 주요 과제 상술

#### 확장성 병목
- 비교할 모델 쌍의 수는 모델 수의 제곱에 비례해 증가하기 때문에, 데이터가 많이 필요
- 순위 알고리즘은 보통 전이성을 가정하나, 전이성 가정이 AI 모델에도 적용되는지는 불분명
- 새로운 모델을 평가 어려움
  - 기존 모델과 새로운 모델 비교해야 하며, 기존 순위에 변경이 발생할 수 있음
- 비공개 모델 평가도 어려움
  - 직접 비교 데이터 수집해 순위표를 만들거나, 공개 순위표 제공업체에 비공개 평가 의뢰해야 함
- 매칭 알고리즘으로써 완화 가능

#### 표준화와 품질 관리의 부재
- LMSYS 챗봇 아레나와 같은 커뮤니티를 통해 비교를 맡길 수 있음
- 표준화와 품질 관리를 강제하기 어려운 단점이 있음
  - 인터넷에 접속할 수 있는 사람이라면 누구나 아무 프롬프트를 이용해 모델 평가할 수 있고, 어떤 응답이 나은지에 대한 기준이 없음
  - 크라우드소싱 방식의 비교는 사용자들이 실제 업무 환경이 아닌 곳에서 모델을 평가
- 표준화를 강제하는 방법으로 사용자가 미리 정해진 프롬프트만 사용하도록 제한할 수 있음
  - 다양한 활용 사례에 대한 모델 능력 평가 불가
- 다른 방법으로 신뢰할 수 있는 평가자만 사용할 수 있음
- 비교 평가를 제품에 통합해 사용자가 작업하는 동안 모델을 평가하도록 할 수도 있음

#### 비교 성능에서 절대 성능으로
- 비교 평가는 모델이 얼마나 좋은지 또는 활용 사례에 충분한지 알려주지 않음
- 비용과 같은 다른 요소들도 고려해야 하는데, 비용 대비 효과 등 분석 어려움

### 3.5.2. 비교 평가의 미래
- 모델이 강력해져 사람의 성능을 뛰어넘게 되어 사람 평가자가 모델 응답에 구체적인 점수를 매기는 게 불가능해질 경우 사용 가능
- 비교 평가는 품질, 즉 사람의 선호도를 파악하는 것을 목표로 함
- 비교 평가는 참조 데이터로 모델을 학습 시키는 것 같은 편법을 쓰기 어려워 상대적으로 조작이 어려움
- 다른 방법으로 알 수 없는 모델 간 성능 차이를 보여줄 수 있음